## The RLMPC code for the clusters, including sending the Ad and Bd matrices.

In [ ]:
%reset

In [ ]:
##IMPORTING LIBRARIES
import time
import struct
import numpy as np
import pandas as pd
import seaborn as sns
from mpi4py import MPI
from itertools import count
from datetime import datetime
from numpy.linalg import inv
import matplotlib.pyplot as plt
from pyModbusTCP.client import ModbusClient
from matplotlib.animation import FuncAnimation

Background of Model Predictive Control

\begin{equation}
%\label{eq:ocp1}
J_{\infty}(x_{k},\textbf{u}_{k}) = \sum_{i=0}^{N-1} \ell (x_{i|k},u_{i|k}) + F({\cdot})
\end{equation}

\begin{equation}
%\label{eq:ocp1}
x_{0|k} = x_{k}
\end{equation}

\begin{equation}
\begin{array}{ccc}
x_{i+1|k}   & =     & f (x_{i|k},u_{i|k}), 0 \leq i \leq N-1    \\
u_{i|k}     & \in   & u, 0 \leq i \leq N-1                      \\
x_{i|k}     & \in   & x, 0 \leq i \leq N-1                      \\
x_{N|k}     & \in   & x_{\Omega}                                \\
\end{array} 
\end{equation}
Find the optimal control sequence $\textbf{u}_{k}^{*}$ by solving the finite-horizon minimization problem

\begin{equation}
\ell (x_{k},u_{k}) = x_{k}^{T}Qx_{k}+u_{k}^{T}Ru_{k}
\end{equation}

\begin{equation}
J(x_{k},u_{k}) = \sum_{i=0}^{3-1} \ell (x_{i|k},u_{i|k}) + F(x_{3|k})
\end{equation}

\begin{equation}
J(x_{k},u_{k}) = \sum_{i=0}^{3-1} (x_{0}^{T}Qx_{0}+u_{0}^{T}Ru_{0})+(x_{1}^{T}Qx_{1}+u_{1}^{T}Ru_{1})+(x_{2}^{T}Qx_{2}+u_{2}^{T}Ru_{2})+ \hat{V}(x_{3|k},W) 
\end{equation}

\begin{equation}
x_{1} = Ax_{0}+Bu_{0}
\end{equation}
\begin{equation}
x_{2} = Ax_{1}+Bu_{1}
\end{equation}
\begin{equation}
x_{3} = Ax_{2}+Bu_{2}
\end{equation}

\begin{equation}
\begin{array}{ccc}
x_{1} &=& Ax_{0}+Bu_{0}  \\
x_{2} &=& A^{2}x_{0}+ABu_{0}+Bu_{1} \\
x_{3} &=& A^{3}x_{0}+A^{2}Bu_{0}+ABu_{1}+Bu_{2} \\
\end{array} 
\end{equation}

\begin{equation}
\begin{array}{ccc}
x_{1} &=& Ax_{0}+Bu_{0}  \\
x_{2} &=& A^{2}x_{0}+ABu_{0}+Bu_{1} \\
x_{3} &=& A^{3}x_{0}+A^{2}Bu_{0}+ABu_{1}+Bu_{2} \\
\end{array} 
\end{equation}

\begin{equation}
ACC = \sum_{i=0}^{k} \ell (x_{i|k},u_{i|k})
\end{equation}

In [ ]:
########################################################################################
## IMPORTING LIBRARIES
########################################################################################
import time
import struct
import numpy as np
import pandas as pd
import seaborn as sns
from mpi4py import MPI
from itertools import count
from numpy.linalg import inv
from datetime import datetime
import matplotlib.pyplot as plt
from pyModbusTCP.client import ModbusClient

########################################################################################
## START MESSAGE PASSING INTERFACE
########################################################################################

comm = MPI.COMM_WORLD
size = comm.Get_size()
rank = comm.Get_rank()

if __name__ == "__main__":
    
    ########################################################################################
    ## INITIALIZE LOCAL VARIABLES
    ########################################################################################
    
    if rank == 0:
        k = ((2 * np.pi) / 1024)         
        
        ## SAMPLE TIME VARIABLES 
        t = 0.05
        Time = 0
        NewTime = t
        latency_history = []
        smoothed_latency = 0 
        
        ## SMOOTHING FACTOR
        alpha = 0.3  
        Latency = []
        RealTime = []
        SimulateTime = []

        ElapsedTime = 0
        servo_position = 0
        encoder_position = 0

        ## FILTER VARIABLES
        Phi = np.array([[1, t], [0, 1]])
        Gamma = np.array([[0], [t]])
        H = np.array([[1, 0], [0, 0]])
        Q = np.array([[1e-5, 0], [0, 1e-3]])
        R = np.array([[1e-4, 0], [0, 1e-2]])
        x1_pos = np.array([[0], [0]])
        x3_pos = np.array([[0], [0]])
        P1_pos = np.array([[0, 0], [0, 0]])
        P3_pos = np.array([[0, 0], [0, 0]])

        ## MOVING AVERAGE
        x_pri1 = [0] * 3
        x_pri3 = [0] * 3
        acel_pri1 = [0] * 20
        acel_pri3 = [0] * 20
        
    else:
        ## INITIALIZES LOCAL VARIABLES
        Q = np.diag([.1, 1e-9, 350,1e-9])
        R = np.array([[0.04]])

    comm.Barrier()
    
    ########################################################################################
    ## LOCAL FUNCTIONS
    ########################################################################################
    
    if rank == 0:
        def DoubleKalmanFilter(x1, x3, x1_pos, x3_pos, P1_pos, P3_pos):
            
            x_pri1.pop(0)
            x_pri3.pop(0)
            x_pri1.append(x1)
            x_pri3.append(x3)    
            
            a1 = (x_pri1[2] - 2*x_pri1[1] + x_pri1[0]) / t**2
            a3 = (x_pri3[2] - 2*x_pri3[1] + x_pri3[0]) / t**2    
            
            acel_pri1.pop(0)
            acel_pri3.pop(0)
            acel_pri1.append(a1)
            acel_pri3.append(a3)
            
            acel_avr1 = sum(acel_pri1) / len(acel_pri1)
            acel_avr3 = sum(acel_pri3) / len(acel_pri3)    
            
            x1_pri = Phi @ x1_pos + Gamma @ np.array([[acel_avr1]])
            x3_pri = Phi @ x3_pos + Gamma @ np.array([[acel_avr3]])
            P1_pri = Phi @ P1_pos @ Phi.T + Q
            P3_pri = Phi @ P3_pos @ Phi.T + Q 
            
            K1 = P1_pri @ H.T @ np.linalg.inv(H @ P1_pri @ H.T + R)
            K3 = P3_pri @ H.T @ np.linalg.inv(H @ P3_pri @ H.T + R)
            
            z1 = np.array([[x1], [0]])
            z3 = np.array([[x3], [0]]) 
            
            x1_pos = x1_pri + K1 @ (z1 - H @ x1_pri)
            x3_pos = x3_pri + K3 @ (z3 - H @ x3_pri)          
            
            P1_pos = P1_pri - K1 @ H @ P1_pri
            P3_pos = P3_pri - K3 @ H @ P3_pri
            
        return x1_pos, x3_pos, P1_pos, P3_pos
    else:
        def Phi(x):
            Phi_function = np.array([
                [x[0][0]], 
                [x[1][0]], 
                [x[2][0]], 
                [x[3][0]], 
                [x[0][0]**2], 
                [x[1][0]**2], 
                [x[2][0]**2], 
                [x[3][0]**2], 
                [x[0][0] * x[1][0]], 
                [x[0][0] * x[2][0]], 
                [x[0][0] * x[3][0]], 
                [x[1][0] * x[2][0]], 
                [x[1][0] * x[3][0]], 
                [x[2][0] * x[3][0]], 
                [x[0][0]**3], 
                [x[1][0]**3], 
                [x[2][0]**3], 
                [x[3][0]**3], 
                [x[0][0]**2 * x[1][0]], 
                [x[0][0]**2 * x[2][0]], 
                [x[0][0]**2 * x[3][0]], 
                [x[1][0]**2 * x[0][0]], 
                [x[1][0]**2 * x[2][0]], 
                [x[1][0]**2 * x[3][0]], 
                [x[2][0]**2 * x[0][0]], 
                [x[2][0]**2 * x[1][0]], 
                [x[2][0]**2 * x[3][0]], 
                [x[3][0]**2 * x[0][0]], 
                [x[3][0]**2 * x[1][0]], 
                [x[3][0]**2 * x[2][0]]
            ])
            return Phi_function

    comm.Barrier()
    
    ########################################################################################
    ## GLOBAL VARIABLE AND FUNCTIONS
    ########################################################################################

    Ad = np.array([
        [0.99874428, 0.03121631, 0.01778894, 0.01548782],
        [0.        , 0.95531125, 0.04201854, 0.13581604],
        [0.        , 0.00541504, 0.61682123, 0.04689394],
        [0.        ,-0.22044499,-0.60307818, 0.82223969]])
    
    Bd = np.array([
        [1.80364346e-05],
        [1.53211718e-04],
        [7.92726805e-05],
        [5.16088775e-04]])
    
    A_bar = np.block([[Ad],[Ad**2],[Ad**3]])
    B_bar = np.block([[Bd, np.zeros((4, 1)), np.zeros((4, 1))],[Ad.dot(Bd), Bd, np.zeros((4, 1))],[(Ad**2).dot(Bd), Ad.dot(Bd), Bd]])
    
    W =  np.zeros((30, 1))
    p = int(len(action) / (size - 1))
    
    action = np.linspace(-98,98, num=199)
    u_bar = []
    for i in range(len(action)):
        for j in range(len(action)):
            for k in range(len(action)):
                u_bar.append(np.array([[action[i]],[action[j]],[action[k]]]))

    comm.Barrier()
    
    ########################################################################################
    ## INPUT THE SYSTEM VARIABLES
    ########################################################################################
    
    if rank == 0:
        c = ModbusClient(host="192.168.15.150", port=502, auto_open=True, auto_close=True)
        
## ====================================================================================================================
        print(f'[INFO] Posicionamento')
        c.write_multiple_registers(10, [int(0)])
        time.sleep(1.5)
        simulation_time = input('[INPUT] Digite o Tempo de Simulação em Segundos: ')
        simulation_time = float(simulation_time)
## ====================================================================================================================

    comm.Barrier()
    
    ########################################################################################
    ## THE PROGAM
    ########################################################################################
    
    if rank == 0:
        ## INITIALIZES LOCAL VARIABLES
        z = []
        ACC = []
        
        u = 0
        J = 0
        
        print("[INFO] Posiciona a Haste proximo a 0°")
        encoder_position = c.read_holding_registers(0, 4)
        x3 = (float(struct.unpack('>h', struct.pack('>H', encoder_position[0]))[0]) * k)

        while x3 < -3 or x3 > 3:
            encoder_position = c.read_holding_registers(0, 4)
            x3 = (float(struct.unpack('>h', struct.pack('>H', encoder_position[0]))[0]) * k)

        print(f"[INFO] Test Started")
        startTime = time.time()
        while ElapsedTime < simulation_time:
            
            ## READING THE STATES OF THE CLP
            encoder_position = c.read_holding_registers(0, 4)
            x1 = (float(struct.unpack('>h', struct.pack('>H', encoder_position[2]))[0]) * 0.0001)
            x3 = (float(struct.unpack('>h', struct.pack('>H', encoder_position[0]))[0]) * k)

            ## ESTIMATE OF STATES
            x1_pos, x3_pos, P1_pos, P3_pos = DoubleKalmanFilter(x1, x3, x1_pos, x3_pos, P1_pos, P3_pos)
            
            ## CREATE STATE VECTOR
            x = np.array([[x1_pos[0][0]],[x1_pos[1][0]],[x3_pos[0][0]],[x3_pos[1][0]]])
            data = {'x': x, 'J': J, 'W': W}
            
            
            ## SENDS THE STATE VECTOR TO THE OTHER PROCESSES 
            ## SERVER 
            comm.send(data, dest=1)
            comm.send(data, dest=2)
            comm.send(data, dest=3)
            comm.send(data, dest=4)
            comm.send(data, dest=5)
            comm.send(data, dest=6)
            comm.send(data, dest=7)
            comm.send(data, dest=8)
            comm.send(data, dest=9)
            comm.send(data, dest=10)
            comm.send(data, dest=11)            
            
            ## NODE 1
            comm.send(data, dest=12)
            comm.send(data, dest=13)
            comm.send(data, dest=14)
            comm.send(data, dest=15)
            comm.send(data, dest=16)
            comm.send(data, dest=17)
            comm.send(data, dest=18)
            comm.send(data, dest=19)
            comm.send(data, dest=20)
            comm.send(data, dest=21)
            comm.send(data, dest=22)
            comm.send(data, dest=23)
            
            ## NODE 2
            comm.send(data, dest=24)
            comm.send(data, dest=25)
            comm.send(data, dest=26)
            comm.send(data, dest=27)
            comm.send(data, dest=28)
            comm.send(data, dest=29)
            comm.send(data, dest=30)
            comm.send(data, dest=31)
            comm.send(data, dest=32)
            comm.send(data, dest=33)
            comm.send(data, dest=34)
            comm.send(data, dest=35)                     
            
            ## STORES THE STATES
            r.rpush('x1', x1,'x3', x3,'u', u)
            
            ## RECEIVES THE CALCULATION WITH THE LOCAL MINIMA OF THE OTHER PROCESSES
            ## SERVER 
            df.append(comm.recv(source=1))
            df.append(comm.recv(source=2))
            df.append(comm.recv(source=3))
            df.append(comm.recv(source=4))
            df.append(comm.recv(source=5))
            df.append(comm.recv(source=6))
            df.append(comm.recv(source=7))
            df.append(comm.recv(source=8))
            df.append(comm.recv(source=9))
            df.append(comm.recv(source=10))
            df.append(comm.recv(source=11))           
            
            ## NODE 1
            df.append(comm.recv(source=12))
            df.append(comm.recv(source=13))
            df.append(comm.recv(source=14))
            df.append(comm.recv(source=15))
            df.append(comm.recv(source=16))
            df.append(comm.recv(source=17))
            df.append(comm.recv(source=18))
            df.append(comm.recv(source=19))
            df.append(comm.recv(source=20))
            df.append(comm.recv(source=21))
            df.append(comm.recv(source=22))
            df.append(comm.recv(source=23))
            
            ## NODE 2
            df.append(comm.recv(source=24))
            df.append(comm.recv(source=25))
            df.append(comm.recv(source=26))
            df.append(comm.recv(source=27))
            df.append(comm.recv(source=28))
            df.append(comm.recv(source=29))
            df.append(comm.recv(source=30))
            df.append(comm.recv(source=31))
            df.append(comm.recv(source=32))
            df.append(comm.recv(source=33))
            df.append(comm.recv(source=34))
            W = comm.recv(source=35)

            ## JOIN ALL DATAFRAMES INTO A SINGLE ONE
            df = pd.concat(df, ignore_index=True)
                        
            ## SELECTS THE GLOBAL MINIMUM
            u = df.loc[df['J'].idxmin()]['u1']

            ## SENDS THE OPTIMAL CONTROL TO THE PLC
            c.write_multiple_registers(10, [struct.unpack('>H', struct.pack('>h', int(u)))[0]])

            ## STORES THE DATA
            z.append(x)
            ACC.append(x0.transpose().dot(Q).dot(x0) + np.array(u[0]).transpose().dot(R).dot(np.array(u[0])))             ## TIME UPDATES
            
            Time += t
            ElapsedTime = time.time() - startTime     ## CALCULATE LATENCY
            
            latency = ElapsedTime - Time
            latency_history.append(latency) ## APPLY EWMA
            
            if len(latency_history) == 1:
                smoothed_latency = latency
            else:
                smoothed_latency = alpha * latency + (1 - alpha) * smoothed_latency 
            
            ## UPDATE THE TIME FOR THE NEXT ITERATION    
            NewTime = t - smoothed_latency  
            
            # COMPENSATE SMOOTHLY ## REGISTRATION 
            Latency.append(latency)
            RealTime.append(ElapsedTime)
            SimulateTime.append(Time) 
            
            ## ADJUSTED WAITING TIME
            time.sleep(max(0.00001, NewTime))

           
        ElapsedTime = round(time.time() - startTime, 2)
        print(f"[INFO] End of Test")

        
    elif rank == 35:
        while True:
            ## RECEIVES THE STATE VECTOR, J AND W OF PROCESS 0
            data = comm.recv(source=0)
            x_bar = data['x_bar']
            J = data['J']
            W = data['W']            
            
            ## INITIALIZES LOCAL VARIABLES
            i = 0
            Flag = 0
            
            while Flag == 0:
                W = W + alpha * ((J-W.transpose().dot(Phi(x_bar[8:12]))) * Phi(x_bar[8:12]))
                delta_W = J - W.transpose().dot(Phi(x_bar[8:12]))
                if delta_W < varepsilon:
                    Flag = 1
                elif i < p:
                    Flag = 1
                else:
                    i += 1 
            ## SENDS W MATRIX TO PROCESS 0
            comm.send(W, dest=0)

    else:
        while True:
            ## RECEIVES THE STATE VECTOR AND W OF PROCESS 0
            data = comm.recv(source=0)
            x_bar = data['x_bar']
            W = data['W'] 
            ## INITIALIZES LOCAL VARIABLES
            best_u = None
            best_J = float('inf')
            
            ## STARTS THE MPC CALCULATION
            for k in range(((rank - 1) * p), ((rank) * p)):
                x_bar = A_bar.dot(x0) + B_bar.dot(u_bar[i])
                J = x0.transpose().dot(Q).dot(x0) + np.array(u_bar[i][0]).transpose().dot(R).dot(np.array(u_bar[i][0])) + x_bar[0:4].transpose().dot(Q).dot(x_bar[0:4]) + np.array(u_bar[i][1]).transpose().dot(R).dot(np.array(u_bar[i][1])) + x_bar[4:8].transpose().dot(Q).dot(x_bar[4:8]) + np.array(u_bar[i][2]).transpose().dot(R).dot(np.array(u_bar[i][2])) + W.transpose().dot(Phi(x_bar[8:12]))
                
                ## STORES THE OPTIMAL MPC VALUES
                if J <= J_pri:
                    J = J_k
                    u = u_bar[i]
                    J_pri = J

            ## CREATES THE DATAFRAME
            df = pd.DataFrame([J_pri,u[0][0],u[1][0],u[2][0]])
            df = df.T
            df.columns  = ['J','u1','u2','u3']
            ## SENDS THE DATAFRAME TO PROCESS 0
            comm.send(df, dest=0)
            



In [ ]:
import time
import struct
import numpy as np
import pandas as pd
from mpi4py import MPI
from pyModbusTCP.client import ModbusClient

# ================= MPI INIT =====================
comm = MPI.COMM_WORLD
size = comm.Get_size()
rank = comm.Get_rank()

# ================= CONSTANTS ===================
t = 0.05
alpha = 0.3
k = ((2 * np.pi) / 1024)
action = np.linspace(-98, 98, num=199)
u_bar = [np.array([[i], [j], [k]]) for i in action for j in action for k in action]
p = len(u_bar) // (size - 1)

# ================= FILTERS & MATRICES ===========

Q = np.diag([.1, 1e-9, 350, 1e-9])
R = np.array([[0.04]])
Ad = np.array([
    [0.99874428, 0.03121631, 0.01778894, 0.01548782],
    [0., 0.95531125, 0.04201854, 0.13581604],
    [0., 0.00541504, 0.61682123, 0.04689394],
    [0., -0.22044499, -0.60307818, 0.82223969]])
Bd = np.array([[1.80364346e-05], [1.53211718e-04], [7.92726805e-05], [5.16088775e-04]])
A_bar = np.block([[Ad], [Ad @ Ad], [Ad @ Ad @ Ad]])
B_bar = np.block([[Bd, np.zeros((4, 1)), np.zeros((4, 1))], [Ad @ Bd, Bd, np.zeros((4, 1))], [(Ad @ Ad) @ Bd, Ad @ Bd, Bd]])

# ================= FUNCTIONS ====================
def double_kalman(x1, x3, x1_pos, x3_pos, P1_pos, P3_pos, x_pri1, x_pri3, acel_pri1, acel_pri3):
    x_pri1.pop(0); x_pri3.pop(0)
    x_pri1.append(x1); x_pri3.append(x3)
    a1 = (x_pri1[2] - 2*x_pri1[1] + x_pri1[0]) / t**2
    a3 = (x_pri3[2] - 2*x_pri3[1] + x_pri3[0]) / t**2
    acel_pri1.pop(0); acel_pri3.pop(0)
    acel_pri1.append(a1); acel_pri3.append(a3)
    acel_avr1 = sum(acel_pri1) / len(acel_pri1)
    acel_avr3 = sum(acel_pri3) / len(acel_pri3)
    x1_pri = Phi_mat @ x1_pos + Gamma * acel_avr1
    x3_pri = Phi_mat @ x3_pos + Gamma * acel_avr3
    P1_pri = Phi_mat @ P1_pos @ Phi_mat.T + Q_kf
    P3_pri = Phi_mat @ P3_pos @ Phi_mat.T + Q_kf
    K1 = P1_pri @ H.T @ np.linalg.inv(H @ P1_pri @ H.T + R_kf)
    K3 = P3_pri @ H.T @ np.linalg.inv(H @ P3_pri @ H.T + R_kf)
    z1 = np.array([[x1], [0]])
    z3 = np.array([[x3], [0]])
    x1_pos = x1_pri + K1 @ (z1 - H @ x1_pri)
    x3_pos = x3_pri + K3 @ (z3 - H @ x3_pri)
    P1_pos = P1_pri - K1 @ H @ P1_pri
    P3_pos = P3_pri - K3 @ H @ P3_pri
    return x1_pos, x3_pos, P1_pos, P3_pos

def send_to_all(data):
    for i in range(1, 36):
        comm.send(data, dest=i)

def recv_from_all():
    df_list = [comm.recv(source=i) for i in range(1, 35)]
    W = comm.recv(source=35)
    return df_list, W

# ================= MAIN ====================
if __name__ == '__main__':
    if rank == 0:
        c = ModbusClient(host="192.168.15.150", port=502, auto_open=True, auto_close=True)
        x1_pos = np.zeros((2,1)); x3_pos = np.zeros((2,1))
        P1_pos = np.zeros((2,2)); P3_pos = np.zeros((2,2))
        x_pri1 = [0]*3; x_pri3 = [0]*3
        acel_pri1 = [0]*20; acel_pri3 = [0]*20
        Time = 0; NewTime = t
        latency_history = []; smoothed_latency = 0
        RealTime = []; SimulateTime = []; Latency = []
        print("[INFO] Posicionamento")
        c.write_multiple_registers(10, [int(0)])
        time.sleep(1.5)
        simulation_time = float(input('[INPUT] Digite o Tempo de Simulação em Segundos: '))
        print("[INFO] Posiciona a Haste proximo a 0°")
        while True:
            x3 = struct.unpack('>h', struct.pack('>H', c.read_holding_registers(0, 4)[0]))[0] * k
            if -3 <= x3 <= 3:
                break
        print("[INFO] Test Started")
        start_time = time.time()
        z = []; ACC = []
        x = np.zeros((4,1)); u = 0; J = 0

        while time.time() - start_time < simulation_time:
            enc = c.read_holding_registers(0, 4)
            x1 = struct.unpack('>h', struct.pack('>H', enc[2]))[0] * 0.0001
            x3 = struct.unpack('>h', struct.pack('>H', enc[0]))[0] * k
            x1_pos, x3_pos, P1_pos, P3_pos = double_kalman(x1, x3, x1_pos, x3_pos, P1_pos, P3_pos, x_pri1, x_pri3, acel_pri1, acel_pri3)
            x = np.array([[x1_pos[0,0]], [x1_pos[1,0]], [x3_pos[0,0]], [x3_pos[1,0]]])
            send_to_all({'x': x, 'J': J, 'W': np.zeros((30,1))})
            df_list, W = recv_from_all()
            df = pd.concat(df_list, ignore_index=True)
            u = df.loc[df['J'].idxmin(), 'u1']
            c.write_multiple_registers(10, [struct.unpack('>H', struct.pack('>h', int(u)))[0]])
            z.append(x)
            ACC.append(x.T @ Q @ x + u**2 * R[0,0])
            Time += t
            elapsed = time.time() - start_time
            latency = elapsed - Time
            latency_history.append(latency)
            smoothed_latency = latency if len(latency_history) == 1 else alpha * latency + (1 - alpha) * smoothed_latency
            NewTime = t - smoothed_latency
            Latency.append(latency); RealTime.append(elapsed); SimulateTime.append(Time)
            time.sleep(max(0.00001, NewTime))
        print("[INFO] End of Test")

    elif rank == 35:
        while True:
            data = comm.recv(source=0)
            x_bar = data['x']
            W = data['W']; J = data['J']
            for _ in range(p):  # Atualização com base em W
                phi = Phi(x_bar[0:4])
                delta_W = J - W.T @ phi
                W += alpha * delta_W * phi
                if abs(delta_W) < 1e-3:
                    break
            comm.send(W, dest=0)

    else:
        while True:
            data = comm.recv(source=0)
            x0 = data['x']; W = data['W']
            best_u = None; best_J = float('inf')
            for i in range((rank - 1) * p, rank * p):
                u = u_bar[i]
                x_bar = A_bar @ x0 + B_bar @ u
                J = (x0.T @ Q @ x0 + u[0]**2 * R + x_bar[0:4].T @ Q @ x_bar[0:4] +
                     u[1]**2 * R + x_bar[4:8].T @ Q @ x_bar[4:8] + u[2]**2 * R +
                     W.T @ Phi(x_bar[8:12]))
                if J < best_J:
                    best_u, best_J = u, J
            df = pd.DataFrame([[best_J, best_u[0,0], best_u[1,0], best_u[2,0]]], columns=['J','u1','u2','u3'])
            comm.send(df, dest=0)

# =============== FEATURES LEFT TO MOVE INTO MODULES ===================
# - Phi(x): Polynomial basis expansion (used by W)
# - Storage (Redis or local CSV)
# - Logging/debug control
